# California Residential Home Price Prediction
# Preprocessing Pipeline

This notebook prepares the cleaned, model-ready dataset from the raw CRMLS MLS data explored in `01_exploration.ipynb`.

Decisions made here are documented with reasoning — each choice is intentional and defensible based on findings from the EDA phase.

In [ ]:
import pandas as pd
import numpy as np
import glob
import os

# Load all CSVs at once
files = glob.glob('/Users/granthbangard/Local Files/Documents/granth/UIUC/IDXExchange/DataSet/CRMLSSold*.csv')
df = pd.concat([pd.read_csv(f, low_memory = False) for f in files], ignore_index = True)

In [3]:
df_sfr = df[(df['PropertyType'] == 'Residential') & (df['PropertySubType'] == 'SingleFamilyResidence')].copy()
print(df_sfr.shape)

(399157, 82)


## Section 2 - Filtering & Date Cutoff

Based on EDA findings, records before June 2023 have very low transaction counts (under 100/month) 
consistent with incomplete data ingestion rather than actual market activity. These records are 
excluded to ensure the model trains on reliable, representative data.

In [4]:
df_sfr['CloseDate'] = pd.to_datetime(df_sfr['CloseDate'], errors='coerce')
df_sfr = df_sfr[df_sfr['CloseDate'] >= '2023-06-01'].copy()
print('Shape after date cutoff:', df_sfr.shape)

Shape after date cutoff: (391673, 82)


## Section 3 - Dropping Useless Columns

Columns are dropped for one of three reasons:
- Completely empty after SFR filter (no usable signal)
- Data leakage (contain information only available after the sale)
- Not relevant to property price prediction (agent/office identifiers, administrative fields)

In [5]:
print(df_sfr[['MLSAreaMajor', 'SubdivisionName', 'TaxAnnualAmount', 
              'AssociationFee', 'BuildingAreaTotal', 'Flooring', 
              'Levels']].isnull().sum())

MLSAreaMajor          56300
SubdivisionName      256056
TaxAnnualAmount      391673
AssociationFee       120219
BuildingAreaTotal    367245
Flooring             139789
Levels                40248
dtype: int64


In [6]:
cols_to_drop = [
    # Leakage
    'ListPrice', 'OriginalListPrice', 'DaysOnMarket', 'PurchaseContractDate',
    # Completely empty
    'ElementarySchoolDistrict', 'MiddleOrJuniorSchoolDistrict',
    'WaterfrontYN', 'BasementYN', 'latfilled', 'lonfilled',
    # Agent/office identifiers
    'ListAgentEmail', 'ListAgentFirstName', 'ListAgentLastName',
    'ListAgentFullName', 'ListOfficeName', 'CoListOfficeName',
    'CoListAgentFirstName', 'CoListAgentLastName',
    'BuyerAgentFirstName', 'BuyerAgentLastName', 'BuyerAgentMlsId',
    'BuyerOfficeName', 'BuyerOfficeAOR', 'CoBuyerAgentFirstName',
    'BuyerAgencyCompensation', 'BuyerAgencyCompensationType',
    'ListAgentAOR', 'BuyerAgentAOR',
    # Administrative/irrelevant
    'ListingKey', 'ListingKeyNumeric', 'ListingId',
    'BusinessType', 'TaxYear', 'BuilderName',
    'LotSizeDimensions', 'AboveGradeFinishedArea', 'BelowGradeFinishedArea',
    'TaxAnnualAmount', 'BuildingAreaTotal', 'SubdivisionName'
]

df_sfr = df_sfr.drop(columns=cols_to_drop)
print('Shape after dropping columns:', df_sfr.shape)
print('Remaining columns:', df_sfr.columns.tolist())

Shape after dropping columns: (391673, 42)
Remaining columns: ['Flooring', 'ViewYN', 'PoolPrivateYN', 'CloseDate', 'ClosePrice', 'Latitude', 'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea', 'FireplacesTotal', 'AssociationFeeFrequency', 'MLSAreaMajor', 'CountyOrParish', 'MlsStatus', 'ElementarySchool', 'AttachedGarageYN', 'ParkingTotal', 'PropertySubType', 'LotSizeAcres', 'YearBuilt', 'StreetNumberNumeric', 'BathroomsTotalInteger', 'City', 'BedroomsTotal', 'ContractStatusChangeDate', 'ListingContractDate', 'StateOrProvince', 'CoveredSpaces', 'MiddleOrJuniorSchool', 'FireplaceYN', 'Stories', 'HighSchool', 'Levels', 'LotSizeArea', 'MainLevelBedrooms', 'NewConstructionYN', 'GarageSpaces', 'HighSchoolDistrict', 'PostalCode', 'AssociationFee', 'LotSizeSquareFeet']


In [9]:
# Remove non-CA records
df_sfr = df_sfr[df_sfr['StateOrProvince'] == 'CA'].copy()

# Drop zero-variance and leakage columns
df_sfr = df_sfr.drop(columns=['MlsStatus', 'StateOrProvince', 
                                'PropertyType', 'PropertySubType',
                                'ContractStatusChangeDate'])

print('Shape:', df_sfr.shape)
print('Remaining columns:', df_sfr.columns.tolist())

Shape: (391649, 37)
Remaining columns: ['Flooring', 'ViewYN', 'PoolPrivateYN', 'CloseDate', 'ClosePrice', 'Latitude', 'Longitude', 'UnparsedAddress', 'LivingArea', 'FireplacesTotal', 'AssociationFeeFrequency', 'MLSAreaMajor', 'CountyOrParish', 'ElementarySchool', 'AttachedGarageYN', 'ParkingTotal', 'LotSizeAcres', 'YearBuilt', 'StreetNumberNumeric', 'BathroomsTotalInteger', 'City', 'BedroomsTotal', 'ListingContractDate', 'CoveredSpaces', 'MiddleOrJuniorSchool', 'FireplaceYN', 'Stories', 'HighSchool', 'Levels', 'LotSizeArea', 'MainLevelBedrooms', 'NewConstructionYN', 'GarageSpaces', 'HighSchoolDistrict', 'PostalCode', 'AssociationFee', 'LotSizeSquareFeet']


In [11]:
df_sfr = df_sfr.drop(columns=[
    'ElementarySchool', 'MiddleOrJuniorSchool', 'HighSchool',
    'AssociationFeeFrequency', 'MainLevelBedrooms', 'Flooring',
    'StreetNumberNumeric', 'LotSizeAcres', 'LotSizeArea',
    'CoveredSpaces', 'ParkingTotal', 'FireplaceYN'
])

print('Shape:', df_sfr.shape)
print('Remaining columns:', df_sfr.columns.tolist())

Shape: (391649, 25)
Remaining columns: ['ViewYN', 'PoolPrivateYN', 'CloseDate', 'ClosePrice', 'Latitude', 'Longitude', 'UnparsedAddress', 'LivingArea', 'FireplacesTotal', 'MLSAreaMajor', 'CountyOrParish', 'AttachedGarageYN', 'YearBuilt', 'BathroomsTotalInteger', 'City', 'BedroomsTotal', 'ListingContractDate', 'Stories', 'Levels', 'NewConstructionYN', 'GarageSpaces', 'HighSchoolDistrict', 'PostalCode', 'AssociationFee', 'LotSizeSquareFeet']


## Section 4 - Outlier Removal

Two outlier thresholds applied based on EDA findings:

- **Lower bound:** Records below $50,000 removed — 110 records with prices this low 
  are almost certainly data errors or partial interest transfers, not real home sales.
- **Upper bound:** Records above $5,000,000 removed — prices above this threshold 
  include clear data anomalies (records up to $989M exist in the raw data) that would 
  distort model training. California's legitimate luxury SFR market is well-represented 
  below this threshold.

In [20]:
print("Shape before removal:", df_sfr.shape)
df_sfr = df_sfr[(df_sfr['ClosePrice'] >= 10000) & (df_sfr['ClosePrice'] <= 5000000)].copy()
print('Shape after outlier removal:', df_sfr.shape)

Shape before removal: (385572, 25)
Shape after outlier removal: (385572, 25)


## Section 5 - Missing Value Treatment

Missing values are handled on a column-by-column basis depending on the nature of the field:

- **Latitude/Longitude:** 193 records missing — recovered via geocoding using UnparsedAddress 
  rather than imputation, as median coordinates would place properties at incorrect locations
- **Numeric features:** Imputed with median — robust to remaining outliers
- **Categorical/Boolean features:** Imputed with mode
- **AssociationFee:** Missing likely means no HOA — imputed with 0

In [21]:
print(df_sfr.isnull().sum())

ViewYN                    38551
PoolPrivateYN             37497
CloseDate                     0
ClosePrice                    0
Latitude                    154
Longitude                   154
UnparsedAddress             368
LivingArea                  118
FireplacesTotal          385572
MLSAreaMajor              56100
CountyOrParish                0
AttachedGarageYN          43894
YearBuilt                   204
BathroomsTotalInteger        67
City                        469
BedroomsTotal                 0
ListingContractDate           0
Stories                   50299
Levels                    38846
NewConstructionYN         37626
GarageSpaces              13334
HighSchoolDistrict        99092
PostalCode                    2
AssociationFee           117005
LotSizeSquareFeet          6604
dtype: int64


In [22]:
df_sfr['GarageSpaces'].median()

np.float64(2.0)

In [23]:
print(df_sfr['MLSAreaMajor'].value_counts().head(20))
print(f"\nUnique values: {df_sfr['MLSAreaMajor'].nunique()}")

MLSAreaMajor
699 - Not Defined                       41809
SRCAR - Southwest Riverside County      21459
252 - Riverside                          5927
LAC - Lancaster                          3658
VIC - Victorville                        3633
248 - Corona                             3439
263 - Banning/Beaumont/Cherry Valley     3379
274 - San Bernardino                     3280
259 - Moreno Valley                      3071
PLM - Palmdale                           2886
264 - Fontana                            2732
APPV - Apple Valley                      2721
HSP - Hesperia                           2439
688 - Rancho Cucamonga                   2384
313 - La Quinta South of HWY 111         2149
BKSF - Bakersfield                       2046
686 - Ontario                            1952
670 - Whittier                           1834
83 - Fullerton                           1613
WHLL - Woodland Hills                    1576
Name: count, dtype: int64

Unique values: 1088


In [24]:
# Drop FireplacesTotal
df_sfr = df_sfr.drop(columns=['FireplacesTotal'])

# Drop rows with missing PostalCode and UnparsedAddress
df_sfr = df_sfr.dropna(subset=['PostalCode', 'UnparsedAddress'])

# Impute with 0
df_sfr['AssociationFee'] = df_sfr['AssociationFee'].fillna(0)
df_sfr['NewConstructionYN'] = df_sfr['NewConstructionYN'].fillna('No')

# Impute with median
for col in ['LivingArea', 'YearBuilt', 'BathroomsTotalInteger', 
            'LotSizeSquareFeet', 'GarageSpaces', 'Stories']:
    df_sfr[col] = df_sfr[col].fillna(df_sfr[col].median())

# Impute with mode
for col in ['ViewYN', 'PoolPrivateYN', 'AttachedGarageYN', 'Levels', 'MLSAreaMajor', 'City']:
    df_sfr[col] = df_sfr[col].fillna(df_sfr[col].mode()[0])

print('Shape:', df_sfr.shape)
print('Remaining nulls:')
print(df_sfr.isnull().sum())

/var/folders/n_/hd4whr8972b54sv4t6_db9d80000gn/T/ipykernel_49221/2682886972.py:18: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_sfr[col] = df_sfr[col].fillna(df_sfr[col].mode()[0])


Shape: (385202, 24)
Remaining nulls:
ViewYN                       0
PoolPrivateYN                0
CloseDate                    0
ClosePrice                   0
Latitude                    34
Longitude                   34
UnparsedAddress              0
LivingArea                   0
MLSAreaMajor                 0
CountyOrParish               0
AttachedGarageYN             0
YearBuilt                    0
BathroomsTotalInteger        0
City                         0
BedroomsTotal                0
ListingContractDate          0
Stories                      0
Levels                       0
NewConstructionYN            0
GarageSpaces                 0
HighSchoolDistrict       98838
PostalCode                   0
AssociationFee               0
LotSizeSquareFeet            0
dtype: int64


In [26]:
df_sfr = df_sfr.drop(columns=['MLSAreaMajor'])

In [27]:
df_sfr.isnull().sum()

ViewYN                       0
PoolPrivateYN                0
CloseDate                    0
ClosePrice                   0
Latitude                    34
Longitude                   34
UnparsedAddress              0
LivingArea                   0
CountyOrParish               0
AttachedGarageYN             0
YearBuilt                    0
BathroomsTotalInteger        0
City                         0
BedroomsTotal                0
ListingContractDate          0
Stories                      0
Levels                       0
NewConstructionYN            0
GarageSpaces                 0
HighSchoolDistrict       98838
PostalCode                   0
AssociationFee               0
LotSizeSquareFeet            0
dtype: int64

In [28]:
df_sfr.shape

(385202, 23)

In [29]:
pip install geopy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [geopy]
Note: you may need to restart the kernel to use updated packages.


In [30]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time

geolocator = Nominatim(user_agent="idx_home_price")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

missing_coords = df_sfr[df_sfr['Latitude'].isnull()].copy()

for idx, row in missing_coords.iterrows():
    try:
        location = geocode(row['UnparsedAddress'])
        if location:
            df_sfr.at[idx, 'Latitude'] = location.latitude
            df_sfr.at[idx, 'Longitude'] = location.longitude
    except Exception as e:
        print(f"Failed for index {idx}: {e}")

print('Remaining missing coords:', df_sfr['Latitude'].isnull().sum())

RateLimiter caught an error, retrying (0/2 tries). Called with (*('2801 Hawthorne Avenue',), **{}).
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/urllib3/connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/urllib3/connection.py", line 571, in getresponse
    httplib_response = super().getresponse()
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/http/client.py", line 1430, in getresponse
    response.begin()
    ~~~~~~~~~~~~~~^^
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/http/client.py", line 331, in begin
    version, status, reason = self._read_status()
                              ~~~~~~~~~~~~~~~~~^^
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/http/client.py", line 292, in _read_status
    line = str(self.

Remaining missing coords: 19


In [33]:
missing_coords = df_sfr[df_sfr['Latitude'].isnull()].copy()
print(f"Still missing: {len(missing_coords)}")
print(missing_coords['UnparsedAddress'].values)

Still missing: 19
['0 SE Corner Santa Fe & 3rd' '0 SW Corner of 2nd & Mission'
 '0 Torres & 1st SEC Street' '0 NE Corner Lobos & 1st Avenue'
 '0 SE Corner of Mission & 1st Avenue' '0 NE Guadalupe & 6th Avenue'
 '0 SE Corner Dolores & 8th Avenue' '4323 Oâ\x80\x99Conner Way'
 '0 SE Corner of Ocean & San Antonio' '0     SE Corner Camino Real & 9th'
 '0 NE Corner of Torres Street & 8th Avenue' '0 Carpenter & 2nd NW Corner'
 '0 SE corner of Guadalupe & 3rd'
 '0 NE Corner of Perry Newberry & 5th Avenue'
 '0 NE Corner of Dolores & 11th Avenue' '0 SE Corner of 2nd & Guadalupe'
 '0 NW Santa Rita & 2nd Avenue' '0 Torres & 8th NW Corner'
 '0 Lincoln & 8th NW Corner Street']


In [34]:
print('Shape before:', df_sfr.shape)
df_sfr = df_sfr.dropna(subset=['Latitude', 'Longitude'])
print('Shape after:', df_sfr.shape)

Shape before: (385202, 23)
Shape after: (385183, 23)


## Section 6 - Feature Engineering

New features derived from existing columns to improve model signal:

- **PropertyAge:** Years since construction at time of sale — more interpretable than raw YearBuilt
- **BedBathRatio:** Bedrooms divided by bathrooms — captures home layout efficiency
- **PricePerSqFt:** Not used as a feature (would cause leakage since it's derived from ClosePrice) — noted here to avoid confusion
- **CloseMonth / CloseYear:** Extracted from CloseDate to capture seasonality and year-over-year trends
- **ListingDuration:** Days between ListingContractDate and CloseDate — measures how long the property was on market before closing

In [41]:
df_sfr["PropertyAge"] = df_sfr["CloseDate"].dt.year - df_sfr["YearBuilt"]
df_sfr['BedBathRatio'] = df_sfr['BedroomsTotal'] / df_sfr['BathroomsTotalInteger'].replace(0, np.nan)
df_sfr["CloseMonth"] = df_sfr["CloseDate"].dt.month
df_sfr["CloseYear"] = df_sfr["CloseDate"].dt.year
df_sfr['ListingContractDate'] = pd.to_datetime(df_sfr['ListingContractDate'], errors='coerce')
df_sfr["ListingDuration"] = (df_sfr["CloseDate"] - df_sfr["ListingContractDate"]).dt.days

df_sfr[['PropertyAge', 'BedBathRatio', 'CloseMonth', 'CloseYear', 'ListingDuration']].describe()

,PropertyAge,BedBathRatio,CloseMonth,CloseYear,ListingDuration
count,385183.000000,385036.000000,385183.000000,385183.000000,385183.000000
mean,49.227902,1.472147,6.623893,2024.410537,71.840844
std,27.248733,0.475300,3.298169,0.931664,66.704026
min,-1.000000,0.000000,1.000000,2023.000000,-31.000000
25%,28.000000,1.000000,4.000000,2024.000000,36.000000
50%,49.000000,1.500000,7.000000,2024.000000,53.000000
75%,69.000000,1.666667,9.000000,2025.000000,86.000000
max,249.000000,7.500000,12.000000,2026.000000,14758.000000


In [42]:
print((df_sfr['ListingDuration'] < 0).sum())
print((df_sfr['ListingDuration'] > 730).sum())

43
179


In [43]:
df_sfr = df_sfr[(df_sfr['ListingDuration'] >= 0) & (df_sfr['ListingDuration'] <= 730)].copy()
print('Shape:', df_sfr.shape)

Shape: (384961, 28)


In [44]:
df_sfr = df_sfr.drop(columns=['ListingContractDate', 'CloseDate'])
print('Shape:', df_sfr.shape)
print('Columns:', df_sfr.columns.tolist())

Shape: (384961, 26)
Columns: ['ViewYN', 'PoolPrivateYN', 'ClosePrice', 'Latitude', 'Longitude', 'UnparsedAddress', 'LivingArea', 'CountyOrParish', 'AttachedGarageYN', 'YearBuilt', 'BathroomsTotalInteger', 'City', 'BedroomsTotal', 'Stories', 'Levels', 'NewConstructionYN', 'GarageSpaces', 'HighSchoolDistrict', 'PostalCode', 'AssociationFee', 'LotSizeSquareFeet', 'PropertyAge', 'BedBathRatio', 'CloseMonth', 'CloseYear', 'ListingDuration']


In [45]:
print(df_sfr.dtypes)

ViewYN                      bool
PoolPrivateYN               bool
ClosePrice               float64
Latitude                 float64
Longitude                float64
UnparsedAddress           object
LivingArea               float64
CountyOrParish            object
AttachedGarageYN            bool
YearBuilt                float64
BathroomsTotalInteger    float64
City                      object
BedroomsTotal            float64
Stories                  float64
Levels                    object
NewConstructionYN         object
GarageSpaces             float64
HighSchoolDistrict        object
PostalCode                object
AssociationFee           float64
LotSizeSquareFeet        float64
PropertyAge              float64
BedBathRatio             float64
CloseMonth                 int32
CloseYear                  int32
ListingDuration            int64
dtype: object


## Section 7 - Encoding Categorical Variables

Categorical and boolean columns are converted to numeric representations for model compatibility:

- **Boolean columns** (ViewYN, PoolPrivateYN, AttachedGarageYN): already True/False, no action needed — XGBoost handles these natively
- **Levels:** Simplified from messy multi-value strings to a clean ordinal (1 = one story, 2 = two story, 3 = three or more)
- **NewConstructionYN:** Dropped — 90% missing after Yes/No mapping, no usable signal
- **CountyOrParish, City, PostalCode, HighSchoolDistrict:** Label encoded — high cardinality columns converted to integer codes

A copy of the preprocessed dataframe is saved as `df_model` before encoding to preserve the option of trying alternative encoding strategies without rerunning the full pipeline.

In [46]:
print(df_sfr['Levels'].value_counts())
print(f"\nCity unique: {df_sfr['City'].nunique()}")
print(f"PostalCode unique: {df_sfr['PostalCode'].nunique()}")
print(f"CountyOrParish unique: {df_sfr['CountyOrParish'].nunique()}")
print(f"HighSchoolDistrict unique: {df_sfr['HighSchoolDistrict'].nunique()}")

Levels
One                               255422
Two                               115771
ThreeOrMore                         6446
MultiSplit                          5086
One,Two                              788
Two,MultiSplit                       522
Two,One                              255
ThreeOrMore,MultiSplit               225
One,MultiSplit                       195
Two,ThreeOrMore                      135
One,ThreeOrMore                       52
MultiSplit,One                        25
One,Two,MultiSplit                    12
Two,ThreeOrMore,MultiSplit             9
One,Two,ThreeOrMore                    8
ThreeOrMore,One                        6
One,Two,ThreeOrMore,MultiSplit         3
Two,MultiSplit,One                     1
Name: count, dtype: int64

City unique: 1145
PostalCode unique: 3614
CountyOrParish unique: 63
HighSchoolDistrict unique: 446


In [47]:
def simplify_levels(val):
    if pd.isna(val): return 1
    if 'Three' in str(val) or 'Multi' in str(val): return 3
    if 'Two' in str(val): return 2
    return 1

df_sfr['Levels'] = df_sfr['Levels'].apply(simplify_levels)

In [48]:
df_sfr['NewConstructionYN'] = df_sfr['NewConstructionYN'].map({'Yes': 1, 'No': 0})

In [50]:
from sklearn.preprocessing import LabelEncoder

df_model = df_sfr.copy()
df_model = df_model.drop(columns=['UnparsedAddress'])

le = LabelEncoder()
for col in ['CountyOrParish', 'City', 'PostalCode', 'HighSchoolDistrict']:
    df_model[col] = df_model[col].fillna('Unknown')
    df_model[col] = le.fit_transform(df_model[col])

print('Shape:', df_model.shape)
print(df_model.dtypes)

Shape: (384961, 25)
ViewYN                      bool
PoolPrivateYN               bool
ClosePrice               float64
Latitude                 float64
Longitude                float64
LivingArea               float64
CountyOrParish             int64
AttachedGarageYN            bool
YearBuilt                float64
BathroomsTotalInteger    float64
City                       int64
BedroomsTotal            float64
Stories                  float64
Levels                     int64
NewConstructionYN        float64
GarageSpaces             float64
HighSchoolDistrict         int64
PostalCode                 int64
AssociationFee           float64
LotSizeSquareFeet        float64
PropertyAge              float64
BedBathRatio             float64
CloseMonth                 int32
CloseYear                  int32
ListingDuration            int64
dtype: object


In [51]:
print(df_model['NewConstructionYN'].value_counts(dropna=False))

NewConstructionYN
NaN    347404
0.0     37557
Name: count, dtype: int64


In [52]:
df_model = df_model.drop(columns=['NewConstructionYN'])
print('Shape:', df_model.shape)

Shape: (384961, 24)
